# Voice Recognition Model – Exploratory Analysis

This notebook walks through the complete pipeline:
1. Data inspection and visualisation
2. Feature extraction (MFCC, Log-Mel spectrogram)
3. Quick model training demo
4. Evaluation and visualisation of results


In [ ]:
import os, sys
# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

import config
print('Config loaded. NUM_CLASSES =', config.NUM_CLASSES)
print('PHONEME_CLASSES =', config.PHONEME_CLASSES)

## 1. Generate synthetic audio and inspect waveforms

In [ ]:
from data.download_data import _generate_synthetic_clip, generate_synthetic_dataset

# Generate dataset if it does not exist yet
generate_synthetic_dataset()
print('Synthetic dataset ready.')

In [ ]:
# Load one sample per class and plot waveforms
import glob
from src.preprocess import load_audio, normalise_audio, pad_or_truncate

fig, axes = plt.subplots(5, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, class_name in enumerate(config.PHONEME_CLASSES):
    files = glob.glob(os.path.join(config.RAW_DATA_DIR, class_name, '*.wav'))
    if not files:
        axes[idx].set_title(f'{class_name} (no file)')
        continue
    audio = load_audio(files[0])
    audio = normalise_audio(audio)
    audio = pad_or_truncate(audio)
    t = np.linspace(0, config.DURATION, len(audio))
    axes[idx].plot(t, audio, linewidth=0.5)
    axes[idx].set_title(class_name)
    axes[idx].set_xlabel('Time (s)')
    axes[idx].set_ylabel('Amplitude')

plt.suptitle('Waveforms – one sample per phoneme class', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'waveforms.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Waveform plot saved.')

## 2. Feature Extraction – MFCC & Log-Mel Spectrogram

In [ ]:
from src.features import extract_mfcc, extract_log_mel_spectrogram, mfcc_to_vector

# Pick the first file from 'vowel_open' as an example
sample_files = glob.glob(os.path.join(config.RAW_DATA_DIR, 'vowel_open', '*.wav'))
sample_audio = load_audio(sample_files[0])
sample_audio = normalise_audio(sample_audio)
sample_audio = pad_or_truncate(sample_audio)

mfcc    = extract_mfcc(sample_audio)
log_mel = extract_log_mel_spectrogram(sample_audio)

print('MFCC shape    :', mfcc.shape)
print('Log-Mel shape :', log_mel.shape)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

img1 = librosa.display.specshow(
    mfcc, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
    x_axis='time', ax=ax1)
ax1.set_title('MFCC')
plt.colorbar(img1, ax=ax1)

img2 = librosa.display.specshow(
    log_mel, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
    x_axis='time', y_axis='mel', ax=ax2)
ax2.set_title('Log-Mel Spectrogram')
plt.colorbar(img2, ax=ax2, format='%+2.0f dB')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'features_sample.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Flat feature vector
vec = mfcc_to_vector(sample_audio)
print('Flat MFCC vector shape:', vec.shape)
print('Expected             :', config.FEATURE_SIZE)

plt.figure(figsize=(12, 2))
plt.plot(vec)
plt.title('Flattened & Normalised MFCC Feature Vector')
plt.xlabel('Feature index')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
from src.model import build_model, get_model_summary

model = build_model()
print(get_model_summary(model))

## 4. Preprocessing Pipeline – Class Distribution

In [ ]:
from src.preprocess import discover_dataset
import collections

file_paths, labels = discover_dataset()
counts = collections.Counter(labels)

classes = [config.PHONEME_CLASSES[i] for i in sorted(counts)]
values  = [counts[i] for i in sorted(counts)]

plt.figure(figsize=(10, 4))
bars = plt.bar(classes, values, color='steelblue', edgecolor='black')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Number of audio files')
plt.title('Dataset Class Distribution')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Total samples:', sum(values))

## 5. Quick Training Demo (optional – takes ~1–2 min)

Uncomment the cell below to run a quick training session.

In [ ]:
# Uncomment to run training
# from src.preprocess import prepare_dataset
# from src.train import train
#
# prepare_dataset()
# metrics = train(epochs=20, batch_size=32)
# print('Training metrics:', metrics)

## 6. Load Saved Results (after training)

In [ ]:
import json

if os.path.exists(config.METRICS_PATH):
    with open(config.METRICS_PATH) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print('No metrics file found – run training first.')

In [ ]:
# Display saved plots if they exist
from IPython.display import Image, display
import os

for plot_path in [config.TRAINING_PLOT, config.CONFUSION_MATRIX]:
    if os.path.exists(plot_path):
        print(f'\n{os.path.basename(plot_path)}')
        display(Image(filename=plot_path))
    else:
        print(f'{plot_path} – not found (run training + evaluation first).')